# M06. Two counts, one object

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tamnd/cpython-internals/blob/main/lessons/m06-two-counts-one-object/m06.ipynb)

Here is the problem in one picture. Two threads each take a reference to the same object at the same moment. Both read the count, both add one, both write it back.

![two threads reading a count of five and both writing six, losing one holder](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m06-two-counts-one-object/diagrams/the-update-that-went-missing.svg)

Two holders, and the object thinks it has one. When the first of them lets go, the count hits zero, the object is destroyed, and the second thread is left holding a pointer into freed memory.

This is why CPython had a lock around the whole interpreter for thirty years. Not because dictionaries are hard to share, but because `count = count + 1` is three machine instructions and the middle one is where the day goes wrong.

## About the source references

Now and then this lesson points at CPython's own source, like this: `Include/object.h:156-167@v3.15.0rc1`.

Read it as three parts: the file, the lines, and the release those line numbers belong to. Sometimes there is a fourth part after a `#`, which is the name of the thing those lines are inside.

Every reference is a link, and every one is checked against the pinned source on each change, so a stale reference fails the build instead of sending you somewhere wrong. You never have to read any of it. The references are there so you can go deeper when you want to, and so you can check that this lesson is not making things up.

## Setup

Colab does not come with the small package these lessons use, so the next cell installs it. If you are running this from a checkout of the repository it is already installed and the cell does nothing.

In [ ]:
import sys

if sys.version_info < (3, 14):
    print("This lesson needs CPython 3.14 or newer.")
    print(f"This runtime is {sys.version.split()[0]}, and the cells below will not run on it.")
else:
    try:
        import pyxray
    except ImportError:
        %pip install -q "pyxray @ git+https://github.com/tamnd/cpython-internals@main#subdirectory=pyxray"
        import pyxray

## Which Python is this

Every cell here runs everywhere, including in a browser, on the interpreter you already have. Nothing in this notebook needs a special build.

Two of the things this lesson is about only exist in a build made without the GIL, and you almost certainly are not running one. Those two arrive as recordings: the program, and what it printed when it ran in the image this project publishes. You can read the program, run the parts of it that work on your own build, and pull the same image if you want to see it for yourself.

Some cells read raw memory through `ctypes`. A wrong address there is a crash rather than an exception, so read what a cell does before you change it.

## Which interpreter is this

In [ ]:
import pyxray

pyxray.show()

## Which build you are on

Start by finding out. There are two ways to ask and they should agree.

sys._is_gil_enabled and the Py_GIL_DISABLED build setting are two ways of asking the same question, and on an ordinary interpreter they say the lock is there

In [ ]:
import sysconfig

disabled = sysconfig.get_config_var("Py_GIL_DISABLED")

print(f"  sys._is_gil_enabled()   {sys._is_gil_enabled()}")
print(f"  Py_GIL_DISABLED         {disabled}")
print()

if sys._is_gil_enabled():
    print("  So there is one lock, and only one thread runs Python at a time.")
    print("  Every count in this notebook is a plain number that only one thread can touch.")
else:
    print("  So there is no lock, and the counts below are split in two.")

`--disable-gil` is a configure flag, not a runtime setting, so this is a property of the binary you are running and there is nothing you can do to it from Python. That build is the [free threaded build](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#free-threaded-build), and everything after the next two sections is about what it had to change.

## What a count looks like today

On the build you are almost certainly running, an object starts with two machine words: the reference count, then a pointer to the type [Include/object.h:139-149@v3.15.0rc1#ob_refcnt](https://github.com/python/cpython/blob/v3.15.0rc1/Include/object.h#L139-L149).

You can read both of them and check your own reading, which is worth doing before trusting anything else read out of memory.

The first word of an object is its reference count and the second is a pointer to its type, and an object with no contents of its own is exactly those two words and nothing else

In [ ]:
import ctypes

WORD = ctypes.sizeof(ctypes.c_void_p)

mine = ["one list, one name"]
AT = id(mine)


def word(at):
    return ctypes.c_ssize_t.from_address(at).value


print(f"  a machine word here is {WORD} bytes")
print(f"  the first word says      {word(AT)}")
print(f"  sys.getrefcount says     {sys.getrefcount(mine)}")
print(f"  the second word is the type: {word(AT + WORD) == id(list)}")
print()
print(f"  so the header is {2 * WORD} bytes, and object() is {sys.getsizeof(object())} bytes")

`sys.getrefcount` reads one higher, because passing the object to it is itself a reference, which is the thing M04 spent a page on.

That count is not something you set. It moves constantly, on its own, as a side effect of ordinary code. Every name that points at the object, every list it goes into, and every function call it is passed to adds one for as long as that lasts.

Passing an object to a function adds one to its count for the duration of the call, and each extra level of nesting adds another

In [ ]:
def one_deep(thing):
    return sys.getrefcount(thing)


def two_deep(thing):
    return one_deep(thing)


print(f"  read from here                {sys.getrefcount(mine)}")
print(f"  read one function call down   {one_deep(mine)}")
print(f"  read two function calls down  {two_deep(mine)}")

holder = [mine] * 5
print(f"  and once it is in a list five times  {sys.getrefcount(mine)}")
del holder
print(f"  and after that list goes away        {sys.getrefcount(mine)}")

That is the volume of the problem. A count going up and down a few times per line of Python, on every object your program touches, forever. Handing that to two threads without a plan is the picture this lesson opened with.

## Playing the race out by hand

You do not need two threads to see what goes wrong, because the failure is arithmetic. Read, add, write, twice over, with the two reads happening before either write.

Two reads of the same count followed by two writes leaves the count one lower than the number of holders, which is exactly one holder too few

In [ ]:
count = 5
print(f"  the object starts with {count} holders")

a_read = count
b_read = count
print(f"  thread A read {a_read} and thread B read {b_read}")

count = a_read + 1
print(f"  thread A wrote back {count}")

count = b_read + 1
print(f"  thread B wrote back {count}")

print()
print(f"  two threads took a reference, so the count should be {5 + 2}")
print(f"  it says {count}, so the object is short by {5 + 2 - count}")
print("  one holder too few means the object gets freed while somebody is still using it")

The usual fix for this is an atomic instruction, which does the read, the add and the write as one indivisible step that no other core can get inside. That works and it is what the free threaded build uses when it has to. The trouble is the price. An atomic add on a value that several cores are reading forces the cache line holding it to bounce between them, and on a hot object that cost is not small.

So the interpreter does not reach for it first. It has three cheaper answers, and the atomic is what is left over.

![four prices for taking a reference, from free to one atomic instruction](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m06-two-counts-one-object/diagrams/three-answers.svg)

The first row is M05, and you have already seen it. The other two are the rest of this lesson.

## Stop counting it at all

The second answer is to pick objects that are read constantly and almost never dropped, and mark them so that nothing bothers counting them. The mark is a number: the object's shared count starts at `PY_SSIZE_T_MAX / 8` [Include/internal/pycore_object.h:24-28@v3.15.0rc1#_Py_REF_DEFERRED](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_object.h#L24-L28), which is a bit over a quintillion.

That is not a count of anything. It is a floor, set so far above zero that no realistic number of decrefs will ever reach it, and if the count can never reach zero then nothing will ever free the object by counting. The cycle collector still can, and it becomes the only thing that ever does. This is [deferred reference counting](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#deferred-reference-counting).

You can work the number out on your own machine, and then check it against the recording below.

The deferred marker is PY_SSIZE_T_MAX divided by 8, and on a build with the GIL nothing is parked there because the whole mechanism is compiled out

In [ ]:
DEFERRED = sys.maxsize // 8

print(f"  sys.maxsize, which is PY_SSIZE_T_MAX  {sys.maxsize}")
print(f"  divided by eight                      {DEFERRED}")
print(f"  and shifted up by two                 {DEFERRED << 2}")
print()
print("  nothing on this build is anywhere near that:")
for label, thing in [
    ("a function", one_deep),
    ("the sys module", sys),
    ("a list", mine),
]:
    count = sys.getrefcount(thing)
    print(f"    {label:16} {count:>10}  past the marker: {count > DEFERRED}")

> **Version note.** These numbers are much smaller on a 32 bit build, including the one running in your browser, because PY_SSIZE_T_MAX is a machine word and there the word is four bytes rather than eight. The recording below came from a 64 bit build, so its marker is the big one.

Now the same question, asked on an interpreter built without the GIL. Everything below came out of the image this project publishes, and the program is the whole program.

on a free threaded build sys.getrefcount returns the deferred marker plus a handful for a top level function, a class, a module and a built in function, and an ordinary number for a list, an instance and a function defined inside another function

What does sys.getrefcount return for an object the interpreter has stopped counting?

```python
"""What the reference count says when nothing is counting it.

M06 says the free threaded build has more than one answer to the cost of counting references,
and that one of them is to stop counting some objects at all. On a real free threaded
interpreter the effect is not subtle: ask for the reference count of an ordinary function and
you get a number close to a quintillion.

That number is a marker, not a count. `_Py_REF_DEFERRED` is `PY_SSIZE_T_MAX / 8`, big enough
that the count can never come back down to zero by accident, so nothing will ever free the
object by counting it. The garbage collector is the only thing that can, and the only thing
that looks.
"""

import sys
import sysconfig

DEFERRED = sys.maxsize // 8

print("the interpreter this ran on")
print()
print(f"    version              {sys.version.split()[0]}")
print(f"    sys._is_gil_enabled  {sys._is_gil_enabled()}")
print(f"    Py_GIL_DISABLED      {sysconfig.get_config_var('Py_GIL_DISABLED')}")
print()

assert not sys._is_gil_enabled()
assert sysconfig.get_config_var("Py_GIL_DISABLED") == 1


def a_function_at_the_top_level():
    pass


class AClassIWrote:
    def a_method(self):
        pass

    @staticmethod
    def a_staticmethod():
        pass


def outer():
    """A function defined inside another one, which is the case that is treated differently."""

    def nested():
        pass

    return nested


print("which objects the interpreter has stopped counting")
print()
for label, obj in [
    ("a list", ["a list"]),
    ("a dict", {}),
    ("a tuple", tuple([1, 2])),
    ("a generator", (n for n in range(3))),
    ("an instance", AClassIWrote()),
    ("a top level function", a_function_at_the_top_level),
    ("a method", AClassIWrote.a_method),
    ("a staticmethod", AClassIWrote.__dict__["a_staticmethod"]),
    ("a nested function", outer()),
    ("a class", AClassIWrote),
    ("the builtin len", len),
    ("the sys module", sys),
]:
    deferred = sys.getrefcount(obj) > DEFERRED
    print(f"    {label:22} {'not counted' if deferred else 'counted normally'}")
print()

print("the actual number, for two of them")
print()
print(f"    sys.getrefcount(a top level function)  {sys.getrefcount(a_function_at_the_top_level)}")
print(f"    sys.getrefcount(the class)             {sys.getrefcount(AClassIWrote)}")
print()
print("and the marker they are both sitting on, which you can work out anywhere")
print()
print(f"    PY_SSIZE_T_MAX             {sys.maxsize}")
print(f"    PY_SSIZE_T_MAX // 8        {DEFERRED}")
print(f"    the same, shifted up by 2  {DEFERRED << 2}")
print()
on_top = sys.getrefcount(a_function_at_the_top_level)
print(f"    the function is the marker plus {on_top - DEFERRED}")
print(f"    the class is the marker plus    {sys.getrefcount(AClassIWrote) - DEFERRED}")
print()

assert sys.getrefcount(a_function_at_the_top_level) > DEFERRED
assert sys.getrefcount(AClassIWrote) > DEFERRED
assert sys.getrefcount(sys) > DEFERRED
assert sys.getrefcount(len) > DEFERRED
assert sys.getrefcount(outer()) < 100, "a nested function is not supposed to be deferred"
assert sys.getrefcount([]) < 100
assert sys.getrefcount(AClassIWrote()) < 100

print("the odd one out is worth a second look")
print()
print("    a function written at the top level of a module gets deferred counting.")
print("    a function written inside another function does not. A nested function has")
print("    probably closed over a variable, and somebody is relying on that variable")
print("    being freed when the function is, rather than whenever the collector next runs.")
print()

import json  # noqa: E402

deferred = sum(1 for value in vars(json).values() if sys.getrefcount(value) > DEFERRED)
total = len(vars(json))
print(f"~ names in the json module that are not counted, out of {total}: {deferred}")
```

```text
the interpreter this ran on

    version              3.15.0rc1
    sys._is_gil_enabled  False
    Py_GIL_DISABLED      1

which objects the interpreter has stopped counting

    a list                 counted normally
    a dict                 counted normally
    a tuple                counted normally
    a generator            counted normally
    an instance            counted normally
    a top level function   not counted
    a method               not counted
    a staticmethod         not counted
    a nested function      counted normally
    a class                not counted
    the builtin len        not counted
    the sys module         not counted

the actual number, for two of them

    sys.getrefcount(a top level function)  1152921504606846977
    sys.getrefcount(the class)             1152921504606846978

and the marker they are both sitting on, which you can work out anywhere

    PY_SSIZE_T_MAX             9223372036854775807
    PY_SSIZE_T_MAX // 8        1152921504606846975
    the same, shifted up by 2  4611686018427387900

    the function is the marker plus 2
    the class is the marker plus    3

the odd one out is worth a second look

    a function written at the top level of a module gets deferred counting.
    a function written inside another function does not. A nested function has
    probably closed over a variable, and somebody is relying on that variable
    being freed when the function is, rather than whenever the collector next runs.

~ names in the json module that are not counted, out of 25: 14
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

Two things in there are worth stopping on.

The first is that the list of objects is not arbitrary. Functions, methods, classes and modules are exactly the things every thread reads on every call and nobody ever drops. A list or an instance is the opposite, and it stays counted.

The second is the nested function. A function written at the top level of a module gets the treatment and one written inside another function does not, and the comment in the source says why: a nested function has probably closed over something, and somebody is relying on that something being freed when the function is rather than whenever the collector next runs [Objects/funcobject.c:224-236@v3.15.0rc1#_PyObject_SetDeferredRefcount](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/funcobject.c#L224-L236). Deferred counting trades prompt destruction for speed, and that trade is not always worth making.

Modules get it in the same one line way [Objects/moduleobject.c:227-234@v3.15.0rc1#track_module](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/moduleobject.c#L227-L234). And there is a public function for turning it on yourself from C [Objects/object.c:2814-2848@v3.15.0rc1#PyUnstable_Object_EnableDeferredRefcount](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/object.c#L2814-L2848), which refuses on anything the collector does not track, for the obvious reason: if nothing counts the object and the collector never looks at it, nothing will ever free it.

## Count it locally, or count it shared

Which leaves everything else. Millions of ordinary objects, all of them counted, none of them safe to count carelessly.

The trick is a bet about how programs behave: most objects are made, used and dropped by one thread and no other thread ever sees them. So give the object a memory of which thread made it, and let that thread count in a plain field with no atomic, while everybody else pays for the atomic on a second field. This is [biased reference counting](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#biased-reference-counting).

![the owning thread adding to the local count against another thread adding to the shared one](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m06-two-counts-one-object/diagrams/where-the-reference-goes.svg)

That needs a bigger header, and the free threaded build has one [Include/object.h:156-167@v3.15.0rc1#ob_ref_shared](https://github.com/python/cpython/blob/v3.15.0rc1/Include/object.h#L156-L167).

![the two word object header next to the seven field free threaded one](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m06-two-counts-one-object/diagrams/two-headers.svg)

Reading the count means adding the two halves together, with one shortcut in front for the immortal case [Include/refcount.h:105-117@v3.15.0rc1#_Py_REFCNT](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L105-L117). And the shared half is not purely a number, because the bottom two bits of it are flags [Include/refcount.h:78-93@v3.15.0rc1#_Py_REF_SHARED](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L78-L93).

![the shared count field with its top bits holding a number and its bottom two holding flags](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m06-two-counts-one-object/diagrams/inside-the-shared-count.svg)

Which means you can decode any shared field you are shown with two operations, and the next cell is a decoder you can point at the numbers in the recording that follows it.

Shifting a shared count field right by two gives the reference count and masking the bottom two bits gives the flags, which is enough to read any of these fields by hand

In [ ]:
SHIFT, FLAGS = 2, 0b11
MEANING = {
    0b00: "nothing has happened yet",
    0b01: "something took a weak reference",
    0b10: "a decref is waiting for the owner",
    0b11: "the two counts have been merged",
}

print("  the raw field        the count  what the bottom two bits say")
for raw in [0, 1, 4, 13, 17, sys.maxsize // 8 << 2]:
    print(f"  {raw:>20}  {raw >> SHIFT:>9}  {MEANING[raw & FLAGS]}")

print()
print("  the last row is the deferred marker again, which is why it fits here:")
print("  deferred counting works by writing an enormous number into this same field")

on a free threaded build a reference taken by the thread that made the object goes into the local count, one taken by any other thread goes into the shared count four at a time, and the reference count you get back is the two added together

Where does a reference go when the thread taking it is not the one that made the object?

```python
"""One object, two reference counts, and which thread gets which.

M06 says the free threaded build splits an object's reference count in two: a plain 32 bit
number that only the owning thread ever writes, and a shared number that everybody else has to
use an atomic for. This is that split, read straight out of memory with ctypes on a real free
threaded interpreter.

Nothing here is a special API. The object header is at `id(x)` and the fields are at fixed
offsets, so the first thing the program does is prove it is reading the right bytes by
checking that the pointer at offset 24 really is the object's type.
"""

import ctypes
import sys
import sysconfig
import threading
import time

#: The free threaded object header, field by field. ob_tid is a thread id or zero, ob_flags
#: and ob_gc_bits are bookkeeping, and the two counts are the point of this program.
TID, FLAGS, GC_BITS, LOCAL, SHARED, TYPE = 0, 8, 11, 12, 16, 24

#: The bottom two bits of ob_ref_shared are flags, not part of the count.
SHIFT, FLAG_MASK = 2, 0x3
FLAG_NAMES = {0x0: "init", 0x1: "maybe weakref", 0x2: "queued", 0x3: "merged"}


def u64(at):
    return ctypes.c_size_t.from_address(at).value


def u32(at):
    return ctypes.c_uint32.from_address(at).value


def u8(at):
    return ctypes.c_uint8.from_address(at).value


def read(at):
    """The three numbers that matter, for the object living at this address."""
    return u64(at + TID), u32(at + LOCAL), ctypes.c_ssize_t.from_address(at + SHARED).value


print("the interpreter this ran on")
print()
print(f"    version              {sys.version.split()[0]}")
print(f"    sys._is_gil_enabled  {sys._is_gil_enabled()}")
print(f"    Py_GIL_DISABLED      {sysconfig.get_config_var('Py_GIL_DISABLED')}")
print()

assert not sys._is_gil_enabled()

watched = ["one list, one name"]
AT = id(watched)

print("proving the offsets before trusting anything read through them")
print()
print(f"    the pointer at offset 24 is the list type:  {u64(AT + TYPE) == id(list)}")
print(f"    ob_gc_bits says the collector tracks it:    {bool(u8(AT + GC_BITS) & 1)}")
print()

assert u64(AT + TYPE) == id(list)


def line(where, at):
    tid, local, shared = read(at)
    count = local + (shared >> SHIFT)
    flag = FLAG_NAMES[shared & FLAG_MASK]
    owned = "yes" if tid else "no"
    print(
        f"    {where:24} owned {owned:3}  local {local:>3}  shared {shared >> SHIFT:>3}"
        f"  flags {flag:<13} total {count}"
    )


print("the same list, held by more names, then borrowed by another thread")
print()
line("just the one name", AT)

box = [watched, watched, watched]
line("three more from here", AT)

seen = []


def borrow():
    """Take three references from a thread that does not own the object, and look."""
    also = [watched] * 3
    seen.append(read(AT))
    del also


worker = threading.Thread(target=borrow)
worker.start()
worker.join()

tid, local, shared = seen[0]
flag = FLAG_NAMES[shared & FLAG_MASK]
print(
    f"    {'three from a worker':24} owned yes  local {local:>3}  shared {shared >> SHIFT:>3}"
    f"  flags {flag:<13} total {local + (shared >> SHIFT)}"
)

line("the worker has finished", AT)
del box
line("back to the one name", AT)
print()

assert seen[0][2] >> SHIFT >= 3, "the worker's references should have gone to the shared count"
assert read(AT)[2] >> SHIFT == 0, "and should have come back off it"

print("which thread owns which object")
print()
made = {}


def make_one():
    mine = ["made over here"]
    made["at"] = id(mine)
    made["tid"] = read(id(mine))[0]
    made["keep"] = mine


second = threading.Thread(target=make_one)
second.start()
second.join()

print(f"    a list made on the main thread has a thread id:   {read(AT)[0] != 0}")
print(f"    a list made on a worker has one too:              {made['tid'] != 0}")
print(f"    and it is a different one:                        {made['tid'] != read(AT)[0]}")
print(f"    None has no owner at all, its ob_tid is:          {read(id(None))[0]}")
print()

assert made["tid"] != read(AT)[0]
assert read(id(None))[0] == 0

print("immortal looks different here")
print()
print(f"    None's ob_ref_local is        {read(id(None))[1]}")
print(f"    which is UINT32_MAX:          {read(id(None))[1] == 2**32 - 1}")
print(f"    sys.getrefcount(None) is      {sys.getrefcount(None)}")
print(f"    which is 3 << 30:             {sys.getrefcount(None) == 3 << 30}")
print()
print("and so does interning, which on this build always means immortal")
print()
built = "".join(["not", "_", "seen", "_", "before"])
IMMORTAL = 2**32 - 1
print(f"    a string you just built:      {read(id(built))[1] == IMMORTAL}")
print(f"    the same after sys.intern:    {read(id(sys.intern(built)))[1] == IMMORTAL}")
print()

#: Py_TPFLAGS_HAVE_GC. A type with this flag gets a collector pre header in front of every
#: instance on an ordinary build, and no pre header at all on this one.
HAVE_GC = 1 << 14

print("what the wider header costs")
print()
for label, obj in [
    ("object()", object()),
    ("an empty tuple", ()),
    ("a one character string", "x"),
    ("an empty list", []),
    ("an empty dict", {}),
]:
    collectable = bool(type(obj).__flags__ & HAVE_GC)
    print(f"    {label:24} {sys.getsizeof(obj):>3} bytes   collectable type: {collectable}")
print()
print("    an object whose type is not collectable pays the whole 16 bytes.")
print("    one whose type is collectable pays nothing, because this build dropped the")
print("    separate collector header and put those bits in the object header instead.")
print()

started = time.monotonic()
holder = []
for _ in range(200000):
    holder.append(watched)
del holder
took = time.monotonic() - started
print(f"~ how long two hundred thousand references took, in seconds: {took:.2f}")
```

```text
the interpreter this ran on

    version              3.15.0rc1
    sys._is_gil_enabled  False
    Py_GIL_DISABLED      1

proving the offsets before trusting anything read through them

    the pointer at offset 24 is the list type:  True
    ob_gc_bits says the collector tracks it:    True

the same list, held by more names, then borrowed by another thread

    just the one name        owned yes  local   1  shared   0  flags init          total 1
    three more from here     owned yes  local   4  shared   0  flags init          total 4
    three from a worker      owned yes  local   4  shared   3  flags maybe weakref total 7
    the worker has finished  owned yes  local   4  shared   0  flags maybe weakref total 4
    back to the one name     owned yes  local   1  shared   0  flags maybe weakref total 1

which thread owns which object

    a list made on the main thread has a thread id:   True
    a list made on a worker has one too:              True
    and it is a different one:                        True
    None has no owner at all, its ob_tid is:          0

immortal looks different here

    None's ob_ref_local is        4294967295
    which is UINT32_MAX:          True
    sys.getrefcount(None) is      3221225472
    which is 3 << 30:             True

and so does interning, which on this build always means immortal

    a string you just built:      False
    the same after sys.intern:    True

what the wider header costs

    object()                  32 bytes   collectable type: False
    an empty tuple            48 bytes   collectable type: True
    a one character string    58 bytes   collectable type: False
    an empty list             56 bytes   collectable type: True
    an empty dict             64 bytes   collectable type: True

    an object whose type is not collectable pays the whole 16 bytes.
    one whose type is collectable pays nothing, because this build dropped the
    separate collector header and put those bits in the object header instead.

~ how long two hundred thousand references took, in seconds: 0.01
```

That ran on Python 3.15.0rc1 in the freethreaded build this project publishes, which is `ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144`. You do not need that build to read the numbers, and you do need it to produce them, which is why this is recorded rather than left as a cell you run. If you want to watch it happen yourself, `docker run --rm -i ghcr.io/tamnd/cpython-internals/cpython:freethreaded@sha256:db72284e3a49f43c38b96bec2baed1380b8348e27ea6f54f6e8d0810b59c3144 python3 -` takes the program on standard input.

Follow the table in the middle of that. One name, local 1. Three more names from the same thread, local 4, and the shared count has not moved. Then three references from a worker thread and the shared count is 3, while local is still 4, so the total is 7. The worker finishes, its three references go, and the shared count is back to 0.

The `flags` column changed from `init` to `maybe weakref` when the worker touched it and stayed that way afterwards. That bit is a hint for later rather than a lie: it means something might have taken a weak reference to this object, so whatever cleans it up has to check. Turning the bit off would mean proving nothing had, which costs more than checking does.

Three more things in that recording are worth naming.

`None` has an `ob_tid` of zero, which is the value for an object no thread owns [Include/object.h:150-154@v3.15.0rc1#_Py_UNOWNED_TID](https://github.com/python/cpython/blob/v3.15.0rc1/Include/object.h#L150-L154). That covers immortal objects and objects whose two counts have been merged, and it is the state you end up in when nobody can be allowed the cheap path.

Immortality is marked differently here. On the build you are running it is a count above two to the power of 31, which M05 showed you. On the free threaded build there is no room for that trick in a 32 bit local count, so the marker is `UINT32_MAX` instead [Include/refcount.h:71-75@v3.15.0rc1#_Py_IMMORTAL_REFCNT_LOCAL](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L71-L75), and `_Py_IsImmortal` tests a different thing depending on which build it is compiled into [Include/refcount.h:126-136@v3.15.0rc1#_Py_IsImmortal](https://github.com/python/cpython/blob/v3.15.0rc1/Include/refcount.h#L126-L136).

And the last two lines of that section are M05's ending, turned inside out. There, `sys.intern` gave you a shared copy and never an immortal one. Here, interning a string makes it immortal every time, because the build cannot afford a shared mutable count on something every thread reads [Objects/unicodeobject.c:14271-14282@v3.15.0rc1#Py_GIL_DISABLED](https://github.com/python/cpython/blob/v3.15.0rc1/Objects/unicodeobject.c#L14271-L14282).

## What the wider header costs

Seven fields instead of two has to be paid for somewhere, and it is worth being precise about where, because the obvious guess is wrong.

![object sizes on a build with the GIL against a build without it](https://raw.githubusercontent.com/tamnd/cpython-internals/main/lessons/m06-two-counts-one-object/diagrams/what-the-header-costs.svg)

The header goes from 16 bytes to 32. But a list, a dict and a tuple are all exactly the same size on both builds, and that is not an accident. On an ordinary build the collector keeps its own [GC pre header](https://github.com/tamnd/cpython-internals/blob/main/GLOSSARY.md#gc-pre-header) of two more words in front of every object whose type it can collect, and the free threaded build does not: those bits live in `ob_gc_bits` inside the object header instead. So `_PyType_PreHeaderSize` adds the collector's header on one build and not the other [Include/internal/pycore_object.h:852-861@v3.15.0rc1#_PyType_PreHeaderSize](https://github.com/python/cpython/blob/v3.15.0rc1/Include/internal/pycore_object.h#L852-L861), and the sixteen bytes the header gained are the sixteen bytes the pre header stopped costing.

The thing that decides which side you land on is a flag on the type, `Py_TPFLAGS_HAVE_GC`. It is not whether a particular object is being tracked right now. A tuple of numbers gets untracked as soon as the collector notices it cannot be part of a cycle, but its type still carries the flag, so it still had the pre header and it still gets the refund.

Which means the bill lands entirely on types the collector never deals with. Integers, strings, bytes, plain `object()` instances. Those get 16 bytes bigger and nothing gives it back.

Whether an object pays for the wider header depends on a flag on its type rather than on whether the collector is tracking that object right now

In [ ]:
import gc

print("  sizes on this build, for comparing against the table in the recording:")
for label, thing in [
    ("object()", object()),
    ("an empty tuple", ()),
    ("a one character string", "x"),
    ("an empty list", []),
    ("an empty dict", {}),
]:
    print(f"    {label:24} {sys.getsizeof(thing):>3} bytes")

print()
print("  and the flag that decides who pays, next to what is tracked right now:")

HAVE_GC = 1 << 14

for label, thing in [
    ("object()", object()),
    ("a one character string", "x"),
    ("a tuple of numbers", (1, 2)),
    ("a list", []),
    ("a dict", {}),
]:
    collectable = bool(type(thing).__flags__ & HAVE_GC)
    print(f"    {label:24} collectable: {collectable!s:5}  tracked now: {gc.is_tracked(thing)}")

> **Version note.** Every size here shrinks on a 32 bit build, including the one in your browser, because a pointer is four bytes there rather than eight. The collectable column is the one that decides who pays, and it never changes. The tracked column can, because a tuple of numbers only gets untracked once the collector has looked at it.

## Try it yourself

Three things.

Take the decoder cell and run it over the raw `ob_ref_shared` values in the second recording. The one for the deferred function is `4611686018427387900`, and shifting it right by two should give you back the marker you computed earlier. Getting the same number two different ways is a good way to be sure you have understood the encoding rather than memorised it.

Work out what the split costs when the bet is wrong. If a thread makes a million objects and hands every one of them to a different thread, every single reference after the first goes through the shared field and pays an atomic, and the local count sits there unused. Write down what you would expect that to look like in the header, then read the second recording's table again and see whether your version matches.

If you want to run the recordings yourself rather than read them, `docker run` the image named at the top of each one and pipe the program in. The digest is in the file, so you will be running the same interpreter these numbers came from, not one that happens to have the same version number.

## What you now know

Two threads incrementing the same count can lose an update, and the object gets freed while somebody is still holding it. That is the reason the GIL existed, and removing the GIL means answering it.

An atomic instruction answers it, and costs enough that the interpreter treats it as the last resort rather than the first.

There are three cheaper answers, in order of how much they save. Immortal objects are never counted at all. Deferred objects have an enormous number written into their shared count, so nothing frees them by counting and only the cycle collector ever can. Everything else gets a count split in two.

Deferred counting goes to top level functions, methods, classes, modules and built in functions. It does not go to nested functions, because a closure holds things somebody wants freed on time.

The split count works because most objects never leave the thread that made them. The object remembers its owner in `ob_tid`, the owner counts in a plain 32 bit field, everybody else counts in a shared field with an atomic, and the reference count is the two added together.

The shared field stores its count shifted up by two, because the bottom two bits are flags: whether something might have taken a weak reference, whether a decref is queued for the owner, and whether the two counts have been merged.

The object header goes from 16 bytes to 32, and a list, a dict and a tuple pay none of it, because the collector's own pre header went away at the same time. The bill lands on types the collector never touches, which is integers, strings, bytes and plain instances of `object`.

## What is next

M07 is the cycle collector, and it follows straight on from here. Two of the three answers in this lesson end with the same sentence: the collector is the only thing that can free this object. That is a lot of responsibility to hand to something you have not looked at yet, so the next lesson looks at it, on the ordinary build first, where it is a generational mark and sweep you can watch run.